# 🎯 burst_scan — two move detectors: will price rise (LONG) or fall (SHORT) by MOVE within WINDOW?

Two **independent** detectors on 1-minute regular-hours bars. At every decision minute each one reads the same
features — built from the current bar and everything before it, back several weeks — and scores how likely it is that:

* **LONG detector** — the price will rise by at least `MOVE_PCT` within `WINDOW_MIN` minutes;
* **SHORT detector** — the price will fall by at least `MOVE_PCT` within `WINDOW_MIN` minutes.

Each is trained only on its own events (walk-forward: every month is scored by a model trained on earlier months).
The question is whether an alert from a detector comes **before a move in its own direction** — not just before a move.

**How to read the scores.** An AUC measures how well a score ranks the right minutes above the others:
**0.5 = coin flip, 1.0 = perfect.** Three per detector:

| score | question | what a real LONG detector needs |
|---|---|---|
| **own AUC** | does it predict its own event (a rise)? | clearly above 0.5 |
| **opposite AUC** | does it predict the OTHER event (a fall) just as well? | clearly BELOW its own AUC |
| **direction AUC** | among the minutes where a move of `MOVE_PCT` happened, does it rank up-moves above down-moves? | clearly above 0.5 |

**The coin-flip market** (`research.sign_surrogate`) is the control: every 1-minute bar of the real prices is
flipped up or down at random. It keeps each bar's size, range, volume and time of day — so a detector of
"something is about to move" works on it just as well — but it contains no direction. A detector only knows
direction if it does clearly better on the real prices than on the coin-flip market.

| § | what |
|---|---|
| 0 | the knobs |
| 1 | the events — how often each happens, by stock and time of day |
| 2 | **the two detectors** — own, opposite and direction AUC, real vs the coin-flip market |
| 3 | **after an alert** — how often the price then moves each way |
| 4 | trading the alerts — before and after costs |
| 5 | **what each detector looks at** — the features it relies on, long vs short |
| 6 | checks and the scorecard |

The study behind this notebook (`ONSET_REPORT.md`) found on 120 names that the existing features predict *that*
price will move, not *which way*; this notebook lets you test that on your own picks and settings.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import backtest as bt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds every computation the five notebooks share; signals() never imports it.
_spec = importlib.util.spec_from_file_location(
    "burst_research", strategies.STRATEGIES_DIR / "burst_scan" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
strat, S = rs.STRAT, rs.S  # the loaded strategy and its module's functions
compose = S["compose"]
SIDES = ("long", "short")
COLOR = {"long": plots._LONG, "short": plots._SHORT}


def below(fig, y=-0.1):
    """Legend under the plot: the default top-right legend sits on the subplot titles."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


from sklearn.inspection import permutation_importance

from trading_simple.data import load_panel

# --- 0 · the knobs --------------------------------------------------------------------------------
SYMBOLS = ["MU", "COIN", "CSCO", "AAL", "ANET"]  # 5 liquid names with 1-min data from 2025-06-18
MOVE_PCT = 0.02  # the move each detector looks for (0.02 = 2%) ...
MOVE_ATR = None  # ... or instead k x each stock's daily ATR% (e.g. 0.5), so calm and wild stocks
#     have equally rare events; None = use MOVE_PCT
WINDOW_MIN = 240  # ... within this many minutes (never past the session's close)
EVERY_MIN = 5  # a decision every N minutes ...
FIRST_MIN, LAST_MIN = 15, 30  # ... from FIRST_MIN after the open to LAST_MIN before the close
MODEL = "gbm"  # "gbm" (gradient boosting) or "lr" (logistic regression)
WARM_MONTHS = 3  # months used only for training before the first scored month
ALERT_Q = (
    0.05  # an alert = the top ALERT_Q of the detector's score (threshold from the PREVIOUS month)
)
SURROGATE_SEED = 0  # the coin-flip market's random signs
START, END = "2025-06-18", "2026-04-30"  # the development window; later months stay unread

## 1 · The events (Table 1, Fig 1)

For every decision minute and stock: from the next minute's open, did the price reach `+MOVE` (a LONG event) or
`−MOVE` (a SHORT event) within the window? Both can happen in the same window.

In [ ]:
panel = load_panel(
    engine, SYMBOLS, "1m", START, END, session="regular", exchanges=settings.exchanges
)
tz = panel.tz
move = rs.atr_move(panel, MOVE_ATR) if MOVE_ATR else MOVE_PCT
rows = rs.detector_rows(panel, every=EVERY_MIN, first_min=FIRST_MIN, last_min=LAST_MIN)
worlds = {"real": panel, "coin-flip": rs.sign_surrogate(panel, SURROGATE_SEED)}
tables, FEATS = {}, None
for name, p in worlds.items():
    feats = rs.detector_features(p)
    tables[name] = rs.detector_table(
        p, rows, feats, rs.move_events(p, rows, move=move, window=WINDOW_MIN)
    )
    FEATS = list(feats)
T = tables["real"]
fees, slip = bt.strategy_costs(strat, panel, settings)
RT = 2 * (fees + slip)
what = f"{MOVE_ATR:.1f} x ATR" if MOVE_ATR else f"{MOVE_PCT:.1%}"
print(
    f"{len(SYMBOLS)} stocks · {T['day'].nunique()} days · {len(T):,} decision minutes · "
    f"{len(FEATS)} features · move {what} within {WINDOW_MIN} min · round trip {1e4 * RT:.0f} bp"
)
table1 = T.groupby("symbol").agg(
    LONG_event=("long", "mean"),
    SHORT_event=("short", "mean"),
    both=("long", lambda s: (s & T.loc[s.index, "short"]).mean()),
    decisions=("long", "size"),
)
display(table1.round(3))

half = (T["minute"] // 30) * 30
fig = go.Figure()
for side, colr in (("long", plots._LONG), ("short", plots._SHORT)):
    r = T.groupby(half)[side].mean()
    fig.add_trace(
        go.Scatter(
            x=[f"{m // 60:02d}:{m % 60:02d}" for m in r.index],
            y=r.to_numpy(),
            mode="lines+markers",
            name=f"{side.upper()} event",
            line={"color": colr},
        )
    )
plots._base(fig, title="Fig 1 · how often a LONG / SHORT event follows, by time of day", height=360)
fig.update_yaxes(tickformat=".0%")
below(fig, -0.2)
fig.show()

## 2 · The two detectors (Table 2, Fig 2)

Each detector is trained walk-forward on its own event only. Read the three AUCs of §0 for each — and compare the real
market with the coin-flip market, which has the same volatility but no direction.

In [ ]:
scores, rows2 = {}, []
for name, t in tables.items():
    s_long = rs.walk_detector(t, FEATS, "long", kind=MODEL, warm_months=WARM_MONTHS)
    s_short = rs.walk_detector(t, FEATS, "short", kind=MODEL, warm_months=WARM_MONTHS)
    scores[name] = (s_long, s_short)
    ok = s_long.notna().to_numpy()
    m, first = t["month"].to_numpy(), t["first"].to_numpy()
    moved = ok & np.isin(first, [1, -1])
    for det, s, own, opp, sign in (
        ("LONG", s_long, "long", "short", 1),
        ("SHORT", s_short, "short", "long", -1),
    ):
        rows2.append(
            {
                "market": name,
                "detector": det,
                "own AUC": rs.monthly_auc(s[ok], t[own][ok], m[ok]),
                "opposite AUC": rs.monthly_auc(s[ok], t[opp][ok], m[ok]),
                "direction AUC": rs.monthly_auc(s[moved], first[moved] == sign, m[moved]),
                "own event rate": t[own][ok].mean(),
            }
        )
table2 = pd.DataFrame(rows2).set_index(["market", "detector"])
display(table2.round(3))

fig = make_subplots(
    rows=1, cols=2, subplot_titles=("LONG detector", "SHORT detector"), shared_yaxes=True
)
for j, det in enumerate(("LONG", "SHORT"), start=1):
    for name, colr in (
        ("real", plots._LONG if det == "LONG" else plots._SHORT),
        ("coin-flip", plots._MUTED),
    ):
        r = table2.loc[(name, det)]
        fig.add_trace(
            go.Bar(
                x=["own", "opposite", "direction"],
                y=[r["own AUC"], r["opposite AUC"], r["direction AUC"]],
                name=f"{name} market, {det}" if name == "real" else "coin-flip market",
                marker_color=colr,
                showlegend=j == 1 or name == "real",
                text=[f"{v:.3f}" for v in (r["own AUC"], r["opposite AUC"], r["direction AUC"])],
                textposition="outside",
            ),
            row=1,
            col=j,
        )
    fig.add_hline(y=0.5, line={"color": plots._INK, "dash": "dot", "width": 1}, row=1, col=j)
fig.update_layout(barmode="group")
fig.update_yaxes(range=[0.4, 0.9])
plots._base(fig, title="Fig 2 · own, opposite and direction AUC (0.5 = coin flip)", height=420)
below(fig, -0.15)
fig.show()

## 3 · After an alert (Table 3, Fig 3)

An **alert** is a minute whose score is in the detector's top `ALERT_Q` — the threshold is taken from the PREVIOUS
month's scores, so nothing peeks at the future. After a LONG alert, how often did the price rise by `MOVE` — and how
often did it fall by `MOVE`? A detector that knows direction raises the first much more than the second.
"Specificity" = (own-event rate − opposite-event rate) after its alerts, minus the same over ALL scored minutes; its t
is clustered by day. (Never compare against the same day's average: that average contains the rest of the day, i.e.
the future — it manufactured a t 8 "VWAP effect" while this notebook was being built.)

In [ ]:
rows3 = []
for name, t in tables.items():
    s_long, s_short = scores[name]
    day = t["day"].to_numpy()
    ok = s_long.notna().to_numpy()
    for det, s, own, opp in (
        ("LONG", s_long, "long", "short"),
        ("SHORT", s_short, "short", "long"),
    ):
        alert = rs.causal_alerts(t, s, ALERT_Q)
        diff = (t[own].astype(float) - t[opp].astype(float)).to_numpy()
        spec = rs.day_t(diff[alert] - diff[ok].mean(), day[alert])
        rows3.append(
            {
                "market": name,
                "detector": det,
                "alerts": int(alert.sum()),
                "own event, all minutes": t[own][ok].mean(),
                "own event after alert": t[own][alert].mean(),
                "opposite event after alert": t[opp][alert].mean(),
                "specificity": spec["mean"],
                "t": spec["t"],
            }
        )
table3 = pd.DataFrame(rows3).set_index(["market", "detector"])
display(table3.round(3))

fig = make_subplots(
    rows=1, cols=2, subplot_titles=("real market", "coin-flip market"), shared_yaxes=True
)
for j, name in enumerate(("real", "coin-flip"), start=1):
    for det, colr in (("LONG", plots._LONG), ("SHORT", plots._SHORT)):
        r = table3.loc[(name, det)]
        fig.add_trace(
            go.Bar(
                x=["own event, all minutes", "own event after alert", "opposite event after alert"],
                y=[
                    r["own event, all minutes"],
                    r["own event after alert"],
                    r["opposite event after alert"],
                ],
                name=f"{det} detector",
                marker_color=colr,
                showlegend=j == 1,
            ),
            row=1,
            col=j,
        )
fig.update_layout(barmode="group")
fig.update_yaxes(tickformat=".0%")
plots._base(
    fig, title="Fig 3 · what follows an alert: its own move — and the opposite move", height=420
)
below(fig, -0.25)
fig.show()

## 4 · Trading the alerts (Table 4)

Each alert as a trade: enter at the next minute's open in the detector's direction, take profit at `MOVE`, stop at the
opposite `MOVE` (a minute touching both is a stop), otherwise exit at the window's last close. Before and after one
round trip of costs; t clustered by day.

In [ ]:
rows4 = []
for name, t in tables.items():
    s_long, s_short = scores[name]
    for det, s, ret in (("LONG", s_long, "ret_long"), ("SHORT", s_short, "ret_short")):
        alert = rs.causal_alerts(t, s, ALERT_Q)
        r = rs.day_t(t[ret].to_numpy()[alert], t["day"].to_numpy()[alert])
        allm = rs.day_t(
            t[ret].to_numpy()[s.notna().to_numpy()], t["day"].to_numpy()[s.notna().to_numpy()]
        )
        rows4.append(
            {
                "market": name,
                "detector": det,
                "trades": r["n"],
                "per trade bp": 1e4 * r["mean"],
                "t": r["t"],
                "every minute bp": 1e4 * allm["mean"],
                "after costs bp": 1e4 * (r["mean"] - RT),
            }
        )
table4 = pd.DataFrame(rows4).set_index(["market", "detector"])
display(table4.round(2))

## 5 · What each detector looks at (Fig 5a, Fig 5b, Table 5)

**Fig 5a** permutation importance: how much each detector's AUC drops when one feature is scrambled (out of sample,
averaged over the scored months) — the features it actually relies on. **Fig 5b** the direction of each feature's
effect in a logistic version of each detector (standardized coefficients): a feature on the diagonal pushes both
detectors the same way (it predicts *movement*); a feature far from it pushes them apart (it would carry
*direction* — if that held out of sample, §2's direction AUC would show it). **Table 5** the features where the two
detectors differ most.

In [ ]:
from scipy.stats import spearmanr

t = tables["real"]
x = t[FEATS].to_numpy(np.float32)
month = t["month"].to_numpy()
imp, coef = {}, {}
for det, y in (("LONG", "long"), ("SHORT", "short")):
    target = t[y].to_numpy().astype(int)
    per, cs = [], []
    for mth in sorted(np.unique(month))[WARM_MONTHS:]:
        fit, test = month < mth, month == mth
        model = S["make_model"](MODEL).fit(x[fit], target[fit])
        per.append(
            permutation_importance(
                model, x[test], target[test], scoring="roc_auc", n_repeats=3, random_state=0
            ).importances_mean
        )
        cs.append(S["make_model"]("lr").fit(x[fit], target[fit])[-1].coef_[0])
    imp[det], coef[det] = np.mean(per, axis=0), np.mean(cs, axis=0)
table5 = pd.DataFrame(
    {
        "importance LONG": imp["LONG"],
        "importance SHORT": imp["SHORT"],
        "coef LONG": coef["LONG"],
        "coef SHORT": coef["SHORT"],
    },
    index=FEATS,
)
top = (
    table5[["importance LONG", "importance SHORT"]]
    .max(axis=1)
    .sort_values(ascending=False)
    .index[:15][::-1]
)
fig = go.Figure()
for det, colr in (("LONG", plots._LONG), ("SHORT", plots._SHORT)):
    fig.add_trace(
        go.Bar(
            y=list(top),
            x=table5.loc[top, f"importance {det}"],
            orientation="h",
            name=f"{det} detector",
            marker_color=colr,
        )
    )
fig.update_layout(barmode="group")
plots._base(
    fig,
    title="Fig 5a · the features each detector relies on (drop in AUC when scrambled)",
    height=560,
)
below(fig, -0.08)
fig.show()

size = table5[["coef LONG", "coef SHORT"]].abs().max(axis=1)
named = set(size.sort_values(ascending=False).index[:15])  # label only the 15 largest effects
fig = go.Figure(
    go.Scatter(
        x=table5["coef LONG"],
        y=table5["coef SHORT"],
        mode="markers+text",
        text=[f if f in named else "" for f in table5.index],
        textposition="top center",
        textfont={"size": 10},
        marker={"color": plots.SERIES[4], "size": 7},
        showlegend=False,
    )
)
lim = float(np.nanmax(np.abs(table5[["coef LONG", "coef SHORT"]].to_numpy()))) * 1.1
fig.add_trace(
    go.Scatter(
        x=[-lim, lim],
        y=[-lim, lim],
        mode="lines",
        name="same effect: movement",
        line={"color": plots._MUTED, "dash": "dot"},
    )
)
fig.add_trace(
    go.Scatter(
        x=[-lim, lim],
        y=[lim, -lim],
        mode="lines",
        name="opposite: direction",
        line={"color": plots._LOSS, "dash": "dot"},
    )
)
plots._base(
    fig,
    title="Fig 5b · each feature's effect in the LONG vs the SHORT detector (standardized)",
    height=620,
)
fig.update_xaxes(title_text="effect in the LONG detector", range=[-lim, lim])
fig.update_yaxes(title_text="effect in the SHORT detector", range=[-lim, lim])
below(fig, -0.12)
fig.show()

rho = spearmanr(imp["LONG"], imp["SHORT"])[0]
print(f"agreement of the two detectors' importance rankings (Spearman): {rho:.2f}")
table5["difference"] = table5["coef LONG"] - table5["coef SHORT"]
display(
    table5.reindex(table5["difference"].abs().sort_values(ascending=False).index).head(10).round(3)
)

## 6 · Checks

In [ ]:
real, surr = table2.loc["real"], table2.loc["coin-flip"]
causal = not rs.causal_alerts(T, scores["real"][0], ALERT_Q)[T["month"] == T["month"].min()].any()
windows_ok = bool(all((T[c].replace(np.inf, 0) <= WINDOW_MIN).all() for c in ("t_long", "t_short")))
vol_ok = bool(
    np.allclose(worlds["coin-flip"].volume.to_numpy(), panel.volume.to_numpy(), equal_nan=True)
)
checks = rs.checks_table(
    [
        ("every event is reached within the window", windows_ok, True, windows_ok),
        (
            "alerts raise nothing in the first scored month (previous-month threshold)",
            causal,
            True,
            causal,
        ),
        ("the coin-flip market keeps the volume", vol_ok, True, vol_ok),
    ]
)
display(checks)
rs.scorecard(
    {
        "notebook": "move_detectors",
        "symbols": SYMBOLS,
        "move": MOVE_ATR or MOVE_PCT,
        "move_is_atr": bool(MOVE_ATR),
        "window_min": WINDOW_MIN,
        "model": MODEL,
        "alert_q": ALERT_Q,
        "auc": {
            f"{w}|{d}": {
                k: float(table2.loc[(w, d), k])
                for k in ("own AUC", "opposite AUC", "direction AUC")
            }
            for w, d in table2.index
        },
        "alert_specificity": {
            f"{w}|{d}": [float(table3.loc[(w, d), "specificity"]), float(table3.loc[(w, d), "t"])]
            for w, d in table3.index
        },
        "trade_bp": {
            f"{w}|{d}": float(table4.loc[(w, d), "per trade bp"]) for w, d in table4.index
        },
        "round_trip_bp": 1e4 * RT,
    }
)